# InfoSeek 1000 Query Images Downloader & Extractor
This notebook downloads the clean 8.3 GB master archive (`infoseek_val_images.tar`) from `BByrneLab/M2KR_Images`, extracts **only your 983 query images**, zips them into `infoseek_val_1000_images.zip` (~80 MB), and backs up both files directly to your **Google Drive**.

### Step 1: Mount Google Drive & Setup Clean Directories

In [ ]:
from google.colab import drive
import os
import shutil

print("Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/OMGM_Data"
os.makedirs(DRIVE_DIR, exist_ok=True)

LOCAL_TAR = "/content/infoseek_val_images.tar"
DRIVE_TAR = os.path.join(DRIVE_DIR, "infoseek_val_images.tar")
REQUIRED_IDS_FILE = "/content/required_image_ids_1000.txt"
OUTPUT_DIR = "/content/infoseek_val_1000_images"
LOCAL_ZIP = "/content/infoseek_val_1000_images.zip"
DRIVE_ZIP = os.path.join(DRIVE_DIR, "infoseek_val_1000_images.zip")

# Clean output dir to avoid any leftovers from previous runs
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Drive backup location: {DRIVE_DIR}")
print(f"Extraction directory:  {OUTPUT_DIR}")

### Step 2: Download Master Archive via Fast CDN & Backup to Drive

In [ ]:
!pip install -q huggingface_hub tqdm

from huggingface_hub import hf_hub_download

# Check if 8.3 GB master archive already exists in Google Drive
if os.path.exists(DRIVE_TAR):
    print(f"Found master archive in Google Drive: {DRIVE_TAR}")
    print("Copying to Colab local SSD for high-speed extraction (~30s)...")
    shutil.copy2(DRIVE_TAR, LOCAL_TAR)
    print("Copied to local SSD!")
else:
    print("Downloading master infoseek_val_images.tar (8.3 GB) via Hugging Face CDN...")
    downloaded_path = hf_hub_download(
        repo_id="BByrneLab/M2KR_Images",
        filename="Infoseek/infoseek_val_images.tar",
        repo_type="dataset",
        local_dir="/content/"
    )
    if downloaded_path != LOCAL_TAR and os.path.exists(downloaded_path):
        shutil.move(downloaded_path, LOCAL_TAR)
    
    print(f"Download complete! Size: {os.path.getsize(LOCAL_TAR) / (1024**3):.2f} GB")
    
    print("Backing up master archive to Google Drive...")
    shutil.copy2(LOCAL_TAR, DRIVE_TAR)
    print("Master archive backed up to Google Drive successfully!")

### Step 3: Extract ONLY the 983 Target Images
Make sure `required_image_ids_1000.txt` is uploaded to `/content/`.

In [ ]:
import tarfile
from tqdm import tqdm

if not os.path.exists(REQUIRED_IDS_FILE):
    raise FileNotFoundError(f"Please upload '{REQUIRED_IDS_FILE}' to Colab root (/content/) first!")

with open(REQUIRED_IDS_FILE, "r") as f:
    target_ids = set(line.strip() for line in f if line.strip())

print(f"Loaded {len(target_ids)} target query image IDs to extract.")

found_ids = set()
print("\nExtracting matching images from master archive...")
with tarfile.open(LOCAL_TAR, "r:") as tar:
    for member in tqdm(tar, desc="Scanning & Extracting"):
        if not member.isfile():
            continue
        
        base_name = os.path.basename(member.name)
        img_id, ext = os.path.splitext(base_name)
        
        if img_id in target_ids:
            dest_path = os.path.join(OUTPUT_DIR, base_name)
            source = tar.extractfile(member)
            if source:
                with open(dest_path, "wb") as out_f:
                    out_f.write(source.read())
                found_ids.add(img_id)

missing_ids = target_ids - found_ids

print("\n" + "=" * 50)
print("EXTRACTION SUMMARY:")
print(f"  - Target requested:  {len(target_ids)}")
print(f"  - Found & extracted: {len(found_ids)}")
print(f"  - Missing:           {len(missing_ids)}")
print("=" * 50)

if missing_ids:
    missing_file = "/content/missing_image_ids.txt"
    with open(missing_file, "w") as f:
        for m in sorted(missing_ids):
            f.write(f"{m}\n")
    print(f"Saved missing IDs to: {missing_file}")

### Step 4: Zip Clean Images & Save to Google Drive

In [ ]:
import zipfile

print(f"\nCompressing {len(found_ids)} clean images into zip...")
with zipfile.ZipFile(LOCAL_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in os.listdir(OUTPUT_DIR):
        zipf.write(os.path.join(OUTPUT_DIR, f), arcname=f)

zip_mb = os.path.getsize(LOCAL_ZIP) / (1024**2)
print(f"Zip created! Size: {zip_mb:.2f} MB")

print(f"Saving zip to Google Drive: {DRIVE_ZIP}...")
shutil.copy2(LOCAL_ZIP, DRIVE_ZIP)
print("SUCCESS! Clean package saved to Google Drive.")
print(f"\nReady for Kaggle: {DRIVE_ZIP}")